In [2]:
import os
import numpy as np
from scipy.io import wavfile
from sklearn import svm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler
import librosa

In [3]:
def extract_features(file_path):
    """Extract MFCC features from a WAV file"""
    y, sr = librosa.load(file_path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    return np.mean(mfcc.T, axis=0)


In [6]:
# Cell 3: Load Dataset
def load_dataset(dataset_path):
    """Load dataset from folder structure"""
    class_folders = sorted([f for f in os.listdir(dataset_path) 
                            if os.path.isdir(os.path.join(dataset_path, f))])
    
    features = []
    labels = []
    
    for label, folder in enumerate(class_folders):
        folder_path = os.path.join(dataset_path, folder)
        for file in os.listdir(folder_path):
            if file.endswith('.wav'):
                file_path = os.path.join(folder_path, file)
                try:
                    feature = extract_features(file_path)
                    features.append(feature)
                    labels.append(label)
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
    
    return np.array(features), np.array(labels), class_folders


In [8]:
# Cell 4: Main Execution
# Set your dataset path
dataset_path = "/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files"  # Change this to your actual dataset path

print("Loading dataset...")
X, y, class_names = load_dataset(dataset_path)
print(f"Loaded {len(X)} samples with {len(class_names)} classes: {class_names}")


Loading dataset...
Loaded 1797 samples with 100 classes: ['1', '10', '100', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '6', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '7', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '8', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '9', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99']


In [9]:
# Cell 5: Train-Test Split and Scaling
# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [1, 5, 10, 20],
    'gamma': [0.01, 0.05, 0.1, 0.5],
    'kernel': ['rbf']
}

grid = GridSearchCV(svm.SVC(), param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation accuracy:", grid.best_score_)

# Use best model
clf = grid.best_estimator_


Best parameters: {'C': 5, 'gamma': 0.05, 'kernel': 'rbf'}
Best cross-validation accuracy: 0.6694420247773907


In [11]:
# Cell 6: Train SVM Classifier
# Create and train SVM classifier
print("\nTraining SVM classifier...")
clf = svm.SVC(kernel='rbf', C=10, gamma=0.1)
clf.fit(X_train, y_train)



Training SVM classifier...


SVC(C=10, gamma=0.1)

In [14]:
# Cell 7: Evaluate Classifier
# Evaluate the classifier
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))
print(f"\nOverall Accuracy: {accuracy:.2%}")


Classification Report:
              precision    recall  f1-score   support

           1       0.00      0.00      0.00         0
          10       0.67      1.00      0.80         4
         100       1.00      1.00      1.00         2
          11       1.00      0.75      0.86         4
          12       1.00      0.33      0.50         6
          13       1.00      1.00      1.00         1
          14       1.00      1.00      1.00         3
          15       0.50      1.00      0.67         3
          16       0.17      1.00      0.29         1
          17       0.80      1.00      0.89         4
          18       0.00      0.00      0.00         3
          19       0.80      1.00      0.89         4
           2       0.00      0.00      0.00         1
          20       0.83      0.83      0.83         6
          21       0.67      0.40      0.50         5
          22       0.67      0.67      0.67         3
          23       1.00      0.50      0.67         4
   

/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/ankitaupadhyaya/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users

In [15]:
# Cell 8: Example Prediction
# Example prediction on a single file
if len(X_test) > 0:
    sample_idx = 7
    sample_feature = X_test[sample_idx].reshape(1, -1)
    predicted_label = clf.predict(sample_feature)[0]
    true_label = y_test[sample_idx]

    print(f"\nExample Prediction:")
    print(f"Predicted: {class_names[predicted_label]} (Class {predicted_label})")
    print(f"Actual: {class_names[true_label]} (Class {true_label})")


Example Prediction:
Predicted: 74 (Class 72)
Actual: 74 (Class 72)


In [16]:
pip install joblib


Note: you may need to restart the kernel to use updated packages.


In [26]:
# Save the trained SVM model and the scaler
import joblib

joblib.dump(clf, 'NEPALI_SVM_model.joblib')
joblib.dump(scaler, 'NEPALI_SVM_scaler.joblib')

print("\nModel and scaler saved successfully!")



Model and scaler saved successfully!


In [28]:
import os
print(os.getcwd())


/Users/ankitaupadhyaya/Project


In [24]:
import numpy as np
import librosa
import joblib

# Load the trained model and scaler
model = joblib.load('NEPALI_SVM_model.joblib')
scaler = joblib.load('NEPALI_SVM_scaler.joblib')

# Your class names in the same order used during training
class_names = sorted([
    folder for folder in os.listdir("/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files")
    if os.path.isdir(os.path.join("/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files", folder))
])

def extract_mfcc(file_path, n_mfcc=40):
    """Extract MFCC features from a .wav file and return the mean vector"""
    y, sr = librosa.load(file_path, sr=None)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean = np.mean(mfcc.T, axis=0)  # Shape: (n_mfcc,)
    return mfcc_mean.reshape(1, -1) 

# Path to your test .wav file
test_file = "/Users/ankitaupadhyaya/Documents/Audio Files/Organized Data/All Files/99/NEP_S4_D(99)_r3.wav"  # Change as needed

# Extract MFCC, scale, and predict
mfcc_features = extract_mfcc(test_file)
scaled_features = scaler.transform(mfcc_features)
predicted_label = model.predict(scaled_features)[0]
predicted_digit = class_names[predicted_label]

print(f"\nPredicted Digit: {predicted_digit} (Class Index: {predicted_label})")



Predicted Digit: 99 (Class Index: 99)
